# AcrPLMEvo Demo (Minimal Smoke Test)

This notebook runs a minimal smoke test for the main experiment entry and then cleans temporary artifacts.

- Environment: `lm-hf`
- Entry script: `/home/nemophila/projects/llm_lora_experiments/scripts/main.py`
- Runtime budget: about 2 minutes

Reviewer note: this demo validates basic execution only. Final reviewer-facing result tables are rebuilt from frozen-evaluation registries (`experiments_frozen_no_lora.csv`, `experiments_frozen.csv`, `experiments_frozen_cross_variant.csv`) via `main.py run-10` + `main.py summary`.
The 10 protocol groups are execution cells (G01-G10), while 6-category reporting merges LoRA/DoRA as one adapter family axis.

In [ ]:
from pathlib import Path
import subprocess

PY = '/home/nemophila/miniconda3/envs/lm-hf/bin/python'
ROOT = Path('/home/nemophila/projects/llm_lora_experiments')
SCRIPTS = ROOT / 'scripts'

print('Python:', PY)
print('Project:', ROOT)

In [ ]:
# Minimal smoke run: single model/cell, tiny samples, 1 epoch
cmd = [
    PY, 'main.py', 'run',
    '--model', 'protbert',
    '--variant', 'lm_only',
    '--seed', '9091',
    '--adapter-type', 'lora',
    '--epochs', '1',
    '--limit-train-batches', '1',
    '--limit-eval-batches', '1',
    '--max-train-samples', '16',
    '--max-valid-samples', '8',
    '--max-test-samples', '8',
    '--save-adapter',
]

result = subprocess.run(cmd, cwd=SCRIPTS, text=True, capture_output=True, timeout=120)
print('Return code:', result.returncode)
print('--- STDOUT (tail) ---')
print('\n'.join(result.stdout.splitlines()[-40:]))
print('--- STDERR (tail) ---')
print('\n'.join(result.stderr.splitlines()[-20:]))

In [ ]:
# Cleanup demo artifacts and rebuild summaries
import pandas as pd
import shutil

run_dir = ROOT / 'results' / 'runs' / 'lora' / 'protbert' / 'lm_only' / 'seed_9091'
if run_dir.exists():
    shutil.rmtree(run_dir)

exp_csv = ROOT / 'results' / 'experiments.csv'
if exp_csv.exists():
    df = pd.read_csv(exp_csv)
    cols = set(df.columns)
    if {'adapter_type', 'model', 'variant', 'seed'}.issubset(cols):
        mask = ~((df['adapter_type'].astype(str) == 'lora')
                 & (df['model'].astype(str) == 'protbert')
                 & (df['variant'].astype(str) == 'lm_only')
                 & (df['seed'].astype(int) == 9091))
        df = df.loc[mask].reset_index(drop=True)
        df.to_csv(exp_csv, index=False)

subprocess.run([PY, 'main.py', 'summary', '--output-root', '../results'], cwd=SCRIPTS, check=True)
print('Cleanup completed.')